In [11]:
import pandas as pd
import torch
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm
import re
import logging
import os # Import os module

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# --- Constants ---
# Define the prompt template using the instructions from the image
# Adapted slightly to ask for punchlines instead of paraphrases.
PROMPT_TEMPLATE = """[INST]
Generate 3 creative and humorous punchlines for the following joke setup.
Format the punchlines as a list numbered as 1., 2., 3. etc.
Answer with only the list of humorous punchlines and nothing else, no exceptions.

Joke Setup: {setup}
[/INST]
"""

# --- Helper Function for Parsing ---
def parse_generated_punchlines(generated_text):
    """
    Parses the generated text to extract three numbered punchlines.
    Attempts to handle variations in model output format.
    """
    # Try extracting using regex for lines starting with number and dot
    punchlines = re.findall(r"^\s*\d+\.\s*(.*)", generated_text, re.MULTILINE)

    # Fallback or refinement: Split by newline and clean up
    if len(punchlines) < 3:
        lines = [line.strip() for line in generated_text.strip().split('\n')]
        potential_punchlines = []
        for line in lines:
            # Remove leading number, dot, and space if present
            cleaned_line = re.sub(r"^\s*\d+\.?\s*", "", line)
            if cleaned_line: # Add only if there's content after cleaning
                potential_punchlines.append(cleaned_line)
        # If regex found some, use them first, then append from fallback
        current_punchlines_set = set(punchlines)
        punchlines.extend(p for p in potential_punchlines if p not in current_punchlines_set)


    # Ensure we have exactly 3, padding with None if necessary
    extracted = [None] * 3
    processed_count = 0
    seen_punchlines = set() # Avoid duplicates if parsing pulls the same line twice
    for p in punchlines:
        if processed_count >= 3:
            break
         # Basic cleaning: remove potential extra quotes or list markers if regex didn't catch them
        clean_p = p.strip().strip('"-*')
        if clean_p and clean_p not in seen_punchlines: # Store None if empty after cleaning or duplicate
            extracted[processed_count] = clean_p
            seen_punchlines.add(clean_p)
            processed_count += 1

    # Log if parsing didn't yield 3 distinct punchlines
    final_count = sum(1 for p in extracted if p is not None)
    if final_count < 3:
        logging.warning(f"Could only parse {final_count} distinct punchlines. Raw text: '{generated_text[:100]}...'")

    return extracted[0], extracted[1], extracted[2]



# --- Main Generation Function ---
def generate_punchlines_instruct_mistral(
    model_path="meta-llama/Llama-3.1-8B-Instruct", # Use an Instruct model
    output_csv_path="llama_instruct_punchlines.csv",
    input_data_path="data/test_data2.csv", # Renamed parameter for clarity
    seed=42,
    batch_size=8, # Reduced batch size for potentially larger models/prompts
    max_new_tokens=150, # Increased token limit for 3 punchlines + formatting
    temperature=0.7,
    top_k=50,
    top_p=0.95
):
    """
    Generates 3 punchlines per setup using a Mistral Instruct model.
    The model is prompted to provide 3 creative and humorous punchlines
    formatted as a numbered list. Results are saved to a CSV.

    Args:
        model_path (str): Path or identifier for the Hugging Face model.
        output_csv_path (str): Path to save the generated results CSV.
        input_data_path (str): Path to the input CSV file containing 'setup' column.
        seed (int): Random seed for reproducibility.
        batch_size (int): Number of setups to process in parallel.
        max_new_tokens (int): Maximum number of tokens to generate for punchlines.
        temperature (float): Sampling temperature for generation.
        top_k (int): Top-k sampling parameter.
        top_p (float): Nucleus sampling parameter.
    """
    # Set random seed
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed) # for multi-GPU
    logging.info(f"Seed set to {seed}")

    # Set the device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    logging.info(f"Using device: {device}")

    # Load model and tokenizer for Instruct model
    logging.info(f"Loading tokenizer from {model_path}...")
    # Ensure padding side is left for generation
    tokenizer = AutoTokenizer.from_pretrained(model_path, padding_side="left")

    logging.info(f"Loading model from {model_path}...")
    # Use bfloat16 for efficiency if supported, and auto device mapping
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16, # Use bf16 if possible
        device_map="auto", # Automatically distribute across available GPUs/CPU
        use_auth_token=""
    )
    model.eval()
    logging.info("Model loaded successfully.")

    # Add padding token if not present
    if tokenizer.pad_token is None:
        logging.info("Tokenizer does not have a pad token. Setting pad_token = eos_token.")
        tokenizer.pad_token = tokenizer.eos_token
        # Important: Configure the model's pad_token_id as well.
        model.config.pad_token_id = model.config.eos_token_id

    # Load input data
    try:
        input_data = pd.read_csv(input_data_path)
        # Ensure 'setup' column exists
        if 'setup' not in input_data.columns:
             raise KeyError("Input CSV must contain a 'setup' column.")
        setups = input_data['setup'].tolist()
        # Handle potential missing ground truth column gracefully
        ground_truth_punchlines = input_data['punchline'].tolist() if 'punchline' in input_data.columns else [None] * len(setups)
        logging.info(f"Loaded {len(setups)} setups from {input_data_path}.")
        if None in ground_truth_punchlines:
             logging.warning("Ground truth 'punchline' column not found or contains missing values in the input CSV.")

    except FileNotFoundError:
        logging.error(f"Error: Input data file not found at {input_data_path}")
        return
    except KeyError as e:
        logging.error(f"Error: {e}")
        return
    except Exception as e:
        logging.error(f"An unexpected error occurred loading data from {input_data_path}: {e}")
        return


    # Initialize results list
    results = []

    # Process in batches
    logging.info(f"Starting generation with batch size {batch_size}...")
    for i in tqdm(range(0, len(setups), batch_size), desc="Generating Punchlines"):
        batch_setups = setups[i:i + batch_size]
        batch_gt_punchlines = ground_truth_punchlines[i:i + batch_size]

        # Create batch of prompts using the template
        batch_prompts = [PROMPT_TEMPLATE.format(setup=s) for s in batch_setups]

        # Tokenize batch
        # Ensure truncation to avoid exceeding model limits
        # Calculate max prompt length dynamically if needed, or use a safe estimate
        # max_model_len = getattr(model.config, 'max_position_embeddings', 2048) # Example
        # max_prompt_len = max_model_len - max_new_tokens - 20 # Leave buffer
        try:
            encoded_input = tokenizer(
                batch_prompts,
                padding=True,
                truncation=True, # Ensure we don't exceed max model length
                max_length=1024, # Set a reasonable max_length for input prompt part
                return_tensors='pt',
                return_attention_mask=True, # Ensure attention mask is returned
            ).to(device) # Move tensors to the correct device
        except Exception as e:
            logging.error(f"Error during tokenization in batch starting at index {i}: {e}")
            # Handle tokenization failure for the batch
            for idx in range(len(batch_setups)):
                 results.append({
                    'setup': batch_setups[idx],
                    'ground_truth_punchline': batch_gt_punchlines[idx],
                    'generated_punchline_1': "TOKENIZATION_ERROR",
                    'generated_punchline_2': "TOKENIZATION_ERROR",
                    'generated_punchline_3': "TOKENIZATION_ERROR",
                    'raw_generated_text': f"Tokenization Error: {e}"
                })
            continue # Skip generation for this batch


        # Generate punchlines for batch
        try:
            with torch.no_grad():
                outputs = model.generate(
                    input_ids=encoded_input['input_ids'],
                    attention_mask=encoded_input['attention_mask'],
                    max_new_tokens=max_new_tokens,
                    temperature=temperature,
                    do_sample=True,
                    top_k=top_k,
                    top_p=top_p,
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                    num_return_sequences=1 # We want one response per prompt (containing 3 punchlines)
                )

            # Decode generated outputs
            prompt_length = encoded_input['input_ids'].shape[1]
            generated_sequences = outputs[:, prompt_length:] # Slice to get only new tokens
            decoded_outputs = tokenizer.batch_decode(generated_sequences, skip_special_tokens=True)

            # --- Store results for batch ---
            for idx, generated_text in enumerate(decoded_outputs):
                # Get original setup and ground truth
                original_setup = batch_setups[idx]
                gt_punchline = batch_gt_punchlines[idx]

                # Parse the generated text to get individual punchlines
                p1, p2, p3 = parse_generated_punchlines(generated_text)

                results.append({
                    'setup': original_setup,
                    'ground_truth_punchline': gt_punchline,
                    'generated_punchline_1': p1,
                    'generated_punchline_2': p2,
                    'generated_punchline_3': p3,
                    'raw_generated_text': generated_text # Optional: store raw output for debugging
                })

        except Exception as e:
            logging.error(f"Error during generation or decoding in batch starting at index {i}: {e}")
            # Add placeholders for failed batch items
            for idx in range(len(batch_setups)):
                 results.append({
                    'setup': batch_setups[idx],
                    'ground_truth_punchline': batch_gt_punchlines[idx],
                    'generated_punchline_1': "GENERATION_ERROR",
                    'generated_punchline_2': "GENERATION_ERROR",
                    'generated_punchline_3': "GENERATION_ERROR",
                    'raw_generated_text': f"Generation/Decoding Error: {e}"
                })

        # Clear GPU cache periodically if using CUDA
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # Convert results list to DataFrame
    result_df = pd.DataFrame(results)

    # Save results
    try:
        # Ensure output directory exists if specified within the path
        output_dir = os.path.dirname(output_csv_path)
        if output_dir and not os.path.exists(output_dir):
            os.makedirs(output_dir)
            logging.info(f"Created output directory: {output_dir}")

        result_df.to_csv(output_csv_path, index=False, encoding='utf-8')
        logging.info(f"Instruct model results saved to {output_csv_path}")
    except Exception as e:
        logging.error(f"Error saving results to CSV '{output_csv_path}': {e}")

In [ ]:

# Define the path to your specific training data
TRAIN_DATA_PATH = "../data/filtered_new_train_data_20k.csv" # Relative path as requested
OUTPUT_FILENAME = "llama_instruct_v01_punchlines_filtered_train_20k_output.csv"

# Check if the input file exists before running
if not os.path.exists(TRAIN_DATA_PATH):
    logging.error(f"FATAL: Input data file not found at the specified path: {TRAIN_DATA_PATH}")
    logging.error("Please ensure the file exists and the path is correct relative to the script location.")
else:
    # Run the generation function with the specified training data path
    generate_punchlines_instruct_mistral(
        model_path="meta-llama/Llama-3.1-8B-Instruct", # Or v0.2, v0.3 etc.
        output_csv_path=OUTPUT_FILENAME,
        input_data_path=TRAIN_DATA_PATH, # Use the specified training data
        batch_size=8, # Adjust based on your GPU memory (8 might be too high for 20k on some GPUs)
        max_new_tokens=150,
        temperature=0.75, # Slightly higher temp for more creativity
        seed=42
    )
    logging.info("Script finished.")